In [22]:
import sys
import torch
import transformers
import pysbd

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PySBD:", pysbd.__version__)
print("Device: CPU")

Python: 3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]
PyTorch: 2.11.0+cu128
Transformers: 4.46.3
PySBD: 0.3.4
Device: CPU


In [23]:
from pathlib import Path
import os

# Set the project directory
PROJECT_DIR = Path.home() / "Downloads" / "GrammarAI_Project"

os.chdir(PROJECT_DIR)

print("Project directory:", Path.cwd())

# Verify source files
print("\nChecking project files:")

print("src folder:", (PROJECT_DIR / "src").exists())

print(
    "model_utils.py:",
    (PROJECT_DIR / "src" / "model_utils.py").exists()
)

print(
    "__init__.py:",
    (PROJECT_DIR / "src" / "__init__.py").exists()
)

Project directory: C:\Users\miraq\Downloads\GrammarAI_Project

Checking project files:
src folder: True
model_utils.py: False
__init__.py: True


In [25]:
import importlib
import src.model_utils as model_utils

# Reload the updated Python module
importlib.reload(model_utils)

from src.model_utils import (
    load_model,
    correct_grammar,
    correct_paragraph,
    validate_output,
    get_word_level_changes,
    render_word_highlights_html)

print("GrammarAI modules imported successfully!")

GrammarAI modules imported successfully!


In [26]:
tokenizer, model, device = load_model()

print("Model:", model.config.architectures)
print("Device:", device)

Model: ['T5ForConditionalGeneration']
Device: cpu


In [27]:
from IPython.display import display, HTML

original = "They was playing cricket when it start raining."

corrected = correct_grammar(original,tokenizer,model)

print("Corrected text:", corrected)

# Display the original with changed words highlighted
highlighted = render_word_highlights_html(original,corrected)

display(HTML(highlighted))

Corrected text: They were playing cricket when it started raining.


In [28]:
from IPython.display import display, HTML

original = "He is engineer."

corrected = correct_grammar(original,tokenizer,model)

print("Original:", original)
print("Corrected:", corrected)

display(HTML(render_word_highlights_html(original,corrected)))

Original: He is engineer.
Corrected: He is an engineer.


In [29]:
paragraph = """
our application uses machine learning to predict delivery time.
the model consider traffic and historical order data.

we evaluate the performance before deploying it to production.
"""

corrected_paragraph = correct_paragraph(paragraph,tokenizer,model)

print("CORRECTED PARAGRAPH:")
print(corrected_paragraph)

print("\nHIGHLIGHTED ORIGINAL:")

display(HTML(render_word_highlights_html(paragraph,corrected_paragraph)))

CORRECTED PARAGRAPH:
Our application uses machine learning to predict delivery time. The model considers traffic and historical order data.

We evaluate the performance before deploying it to production.

HIGHLIGHTED ORIGINAL:


In [31]:
from IPython.display import display, HTML

original = (
    "The model considers traffic weather "
    "and historical order data.")

corrected = (
    "The model considers traffic, weather, "
    "and historical order data.")

display(HTML(render_word_highlights_html(original,corrected)))

In [32]:
from src.model_utils import get_text_level_changes

from src.correction_manager import (
    CorrectionManager,
    CorrectionManagerError,
    CorrectionConflictError)

print("Correction manager imported successfully!")

Correction manager imported successfully!


In [34]:
original = "They was playing cricket when it start raining."

model_corrected = "They were playing cricket when it started raining."

# Generate correction suggestions
changes = get_text_level_changes(original,model_corrected)

# Initialize correction manager
manager = CorrectionManager(original,changes)

# Inspect suggestions
for suggestion in manager.suggestions:
    print(suggestion)

{'kind': 'word', 'start': 5, 'end': 8, 'original': 'was', 'suggestion': 'were', 'change_type': 'replace', 'id': 1, 'status': 'pending'}
{'kind': 'word', 'start': 33, 'end': 38, 'original': 'start', 'suggestion': 'started', 'change_type': 'replace', 'id': 2, 'status': 'pending'}


In [35]:
manager.accept(1)
manager.reject(2)

print("Review summary:")
print(manager.summary())

final_text = manager.apply_accepted()

print("\nFinal text:")
print(final_text)

Review summary:
{'total': 2, 'pending': 0, 'accepted': 1, 'rejected': 1}

Final text:
They were playing cricket when it start raining.


In [36]:
manager.reset_all()

manager.accept_all()

final_text = manager.apply_accepted()

print(final_text)

They were playing cricket when it started raining.


In [37]:
import importlib

import src.model_utils as model_utils
import src.correction_manager as correction_manager

# Reload the latest source files
importlib.reload(model_utils)
importlib.reload(correction_manager)

# Import the required functions and classes
from src.model_utils import (
    correct_grammar,
    correct_paragraph,
    get_text_level_changes,
    render_word_highlights_html,
    validate_output
)

from src.correction_manager import (
    CorrectionManager,
    CorrectionManagerError,
    CorrectionConflictError
)

from IPython.display import display, HTML
import pandas as pd

print("GrammarAI modules loaded successfully!")

GrammarAI modules loaded successfully!


In [39]:
original_text = """
our application use machine learning to predict delivery time.
the model consider traffic weather and historical order data.

we evaluates the performance before deployment.
"""

# Step 1: Generate corrected paragraph using Vennify T5
model_corrected = correct_paragraph(original_text,tokenizer,model)

print("ORIGINAL TEXT:")
print(original_text)

print("\nMODEL CORRECTED TEXT:")
print(model_corrected)

ORIGINAL TEXT:

our application use machine learning to predict delivery time.
the model consider traffic weather and historical order data.

we evaluates the performance before deployment.


MODEL CORRECTED TEXT:
Our application uses machine learning to predict delivery time. The model considers traffic weather and historical order data.

We evaluate the performance before deployment.


In [40]:
changes = get_text_level_changes(original_text,model_corrected)

print("Total suggestions detected:", len(changes))

Total suggestions detected: 5


In [41]:
manager = CorrectionManager(original_text,changes)

print("\nCorrection manager initialized.")
print("Review summary:", manager.summary())


Correction manager initialized.
Review summary: {'total': 5, 'pending': 5, 'accepted': 0, 'rejected': 0}


In [43]:
suggestions_df = pd.DataFrame(manager.suggestions)

display(
    suggestions_df[
        [
            "id",
            "kind",
            "original",
            "suggestion",
            "change_type",
            "status"]])

,id,kind,original,suggestion,change_type,status
0,1,word,our,Our,replace,pending
1,2,word,use,uses,replace,pending
2,3,word,the,The,replace,pending
3,4,word,consider,considers,replace,pending
4,5,word,we evaluates,We evaluate,replace,pending


In [44]:
highlighted_text = render_word_highlights_html(original_text,model_corrected)

display(HTML(highlighted_text))

In [45]:
# Accept suggestion 1
manager.accept(1)

# Reject suggestion 2
manager.reject(2)

# Review current statuses
print(manager.summary())

for suggestion in manager.suggestions:
    print(
        suggestion["id"],
        suggestion["original"],
        "->",
        suggestion["suggestion"],
        "|",
        suggestion["status"])

{'total': 5, 'pending': 3, 'accepted': 1, 'rejected': 1}
1 our -> Our | accepted
2 use -> uses | rejected
3 the -> The | pending
4 consider -> considers | pending
5 we evaluates -> We evaluate | pending


In [46]:
partially_corrected = manager.apply_accepted()

print("ORIGINAL TEXT:")
print(original_text)

print("\nUSER-APPROVED TEXT:")
print(partially_corrected)

ORIGINAL TEXT:

our application use machine learning to predict delivery time.
the model consider traffic weather and historical order data.

we evaluates the performance before deployment.


USER-APPROVED TEXT:

Our application use machine learning to predict delivery time.
the model consider traffic weather and historical order data.

we evaluates the performance before deployment.



In [47]:
accept_all_manager = CorrectionManager(original_text,changes)

accept_all_manager.accept_all()

fully_corrected = accept_all_manager.apply_accepted()

print("ACCEPTED CORRECTIONS:")
print(fully_corrected)

print("\nSTATUS:")
print(accept_all_manager.summary())

ACCEPTED CORRECTIONS:

Our application uses machine learning to predict delivery time.
The model considers traffic weather and historical order data.

We evaluate the performance before deployment.


STATUS:
{'total': 5, 'pending': 0, 'accepted': 5, 'rejected': 0}


In [48]:
reject_all_manager = CorrectionManager(original_text,changes)

reject_all_manager.reject_all()

unchanged_text = reject_all_manager.apply_accepted()

print("REJECTED ALL CORRECTIONS:")
print(unchanged_text)

print("\nOriginal preserved:", unchanged_text == original_text)

REJECTED ALL CORRECTIONS:

our application use machine learning to predict delivery time.
the model consider traffic weather and historical order data.

we evaluates the performance before deployment.


Original preserved: True


In [49]:
original_punctuation = "The model considers traffic weather."

corrected_punctuation = ("The model considers traffic, weather.")

punctuation_changes = get_text_level_changes(original_punctuation,corrected_punctuation)

punctuation_manager = CorrectionManager(original_punctuation,punctuation_changes)

punctuation_manager.accept_all()

print(punctuation_manager.apply_accepted())

The model considers traffic, weather.


In [50]:
validation = validate_output(original_text,fully_corrected)

print("=" * 70)
print("GRAMMARAI FINAL REVIEW")
print("=" * 70)

print("\nOriginal word count:",
      len(original_text.split()))

print("Corrected word count:",
      len(fully_corrected.split()))

print("\nCorrection summary:")
print(accept_all_manager.summary())

print("\nStructural validation:")
print(validation)

print("\nFINAL APPROVED TEXT:")
print(fully_corrected)

GRAMMARAI FINAL REVIEW

Original word count: 24
Corrected word count: 24

Correction summary:
{'total': 5, 'pending': 0, 'accepted': 5, 'rejected': 0}

Structural validation:
{'is_usable': True, 'errors': [], 'warnings': [], 'original_word_count': 24, 'corrected_word_count': 24}

FINAL APPROVED TEXT:

Our application uses machine learning to predict delivery time.
The model considers traffic weather and historical order data.

We evaluate the performance before deployment.

